In [34]:
#https://maykosilva.com/blog/projeto-pratico-de regressao-com-python-e-scikit-learn/
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
#Carregando arquivos
df = pd.read_csv('reviews.csv')
df.head()

In [ ]:
#Verificar valores faltando
print(df.isnull().sum())

#Plotar histogramas das colunas numéricas
numeric_columns = df.select_dtypes(include=[np.number])

num_subplots = len(numeric_columns.columns)
num_rows = (num_subplots + 1) // 2
num_cols = 2

fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, num_rows*4))
fig.suptitle('Histogramas das colunas numéricas', fontsize=16)

for i, column in enumerate(numeric_columns.columns):
  row = i // num_cols
  col = i % num_cols
  ax = axes[row, col]
  ax.hist(numeric_columns[column], bins=20)
  ax.set_title(column)
  ax.set_xlabel('Valor')
  ax.set_ylabel('Frequência')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("Estatisticas prescritivas")
print(df.describe())

In [ ]:
# Verificar a coluna "Classificação"
df['Classificacao'] = pd.to_numeric(df['Classificacao'], errors='coerce')
print("Valores inválidos na coluna 'Classificação':", df['Classificacao'].isnull().sum())

# Verificar a coluna "Preço"
df['Preco'] = pd.to_numeric(df['Preco'], errors='coerce')
print("Valores inválidos na coluna 'Preco':", df['Preco'].isnull().sum())

# Verificar a coluna "Tamanho"
df['Tamanho'] = pd.to_numeric(df['Tamanho'], errors='coerce')
print("Valores inválidos na coluna 'Tamanho':", df['Tamanho'].isnull().sum())

# Verificar a coluna "Dias desde a ultima Atualizacao"
df['Dias desde a ultima Atualizacao'] = pd.to_numeric(df['Dias desde a ultima Atualizacao'], errors='coerce')
print("Valores inválidos na coluna 'Dias desde a ultima Atualizacao':", df['Dias desde a ultima Atualizacao'].isnull().sum())


In [ ]:
# Para mudar a versão original do df e nao criar uma copia
df.dropna(inplace=True)

# Selecionar linhas com valores não-negativos para as colunas "Dias desde a ultima Atualizacao" e "Classificacao"
mask = (df['Dias desde a ultima Atualizacao'] >= 0) & (df['Classificacao'] >= 0)
df = df[mask]

# Verificar o número de linhas restantes

print("Número de linhas após a remoção:", df.shape[0])

In [17]:
X = df.drop('Classificacao', axis=1)
y = df['Classificacao']

X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(X, y, test_size=0.2, random_state=0)


In [ ]:

# Remover a coluna "Categoria" antes de calcular a matriz de correlação
X_treinamento_numeric = X_treinamento.drop('Categoria', axis=1)

# Converter as colunas relevantes para tipos numéricos
numeric_columns = ['No de Reviews', 'No de Instalacoes', 'Tamanho', 'Preco', 'Dias desde a ultima Atualizacao']
X_treinamento_numeric[numeric_columns] = X_treinamento_numeric[numeric_columns].apply(pd.to_numeric, errors='coerce')

# Calcular a matriz de correlação
correlation_matriz = X_treinamento_numeric.corr()
print("\nMatriz de Correlação: ")
print(correlation_matriz)

In [ ]:

columns_to_plot = ['No de Reviews', 'No de Instalacoes', 'Tamanho', 'Preco', 'Dias desde a ultima Atualizacao']

figs, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 10))
fig.suptitle('Gráficos de Dispersão: Classificacao vc Outras Colunas', fontsize=16)
fig.tight_layout(pad=5.0)

for i, column in enumerate(columns_to_plot):
  row = i // 3
  col = i % 3

  axes[row, col].scatter(X_treinamento_numeric[column], y_treinamento)
  axes[row, col].set_title(f'Classificacao vc {column}')
  axes[row, col].set_xlabel(column)
  axes[row, col].set_ylabel('Classificacao')

plt.show()


In [ ]:
# Declarar as variáveis num_col e cat_col
num_col = ['No de Reviews', 'No de Instalacoes', 'Tamanho', 'Preco', 'Dias desde a ultima Atualizacao']
cat_col = ['Categoria']

# Substituir os valores ausentes na coluna "Tamanho" pela média
imp = SimpleImputer(strategy='mean')
tf_num = imp.fit_transform(X_treinamento[num_col])

# Escalar as colunas numéricas
scaler = StandardScaler()
tf_num = scaler.fit_transform(tf_num)

#Codificar a coluna "Categoria" usando a codificacao one-hot
ohe = OneHotEncoder(sparse_output=False, drop='first')
tf_cat = ohe.fit_transform(X_treinamento[cat_col])

#Concatenar os arrays tf_num e tf_cat ao longo do eixo 1
X_treinamento_transformado = np.concatenate((tf_num, tf_cat), axis=1)

#Verificar o resultado imprimindo o primeiro exemplo transformado
print("Exemplo transformado:")
print(X_treinamento_transformado[0])

In [ ]:
# Instanciar o objeto LinearRegression
model = LinearRegression()

# Treinar o modelo usando os dados de treinamento transformados
model.fit(X_treinamento_transformado, y_treinamento)

# Imprimir os coefientes (coef_) e o intercepto (intercept_)
print("Coeficientes:", model.coef_)
print("Intercepto:", model.intercept_)

In [ ]:
# Avaliação no conjunto de dados de treinamento
y_pred_treinamento = model.predict(X_treinamento_transformado)

mse = mean_squared_error(y_treinamento, y_pred_treinamento)
rmse_treinamento = np.sqrt(mse)
r2_treinamento = r2_score(y_treinamento, y_pred_treinamento)

print("RMSE (treinamento):", rmse_treinamento)
print("R (Treinamento):", r2_treinamento)

# Transformação dos dados de teste
tf_num_teste = imp.transform(X_teste[num_col])
tf_num_teste = scaler.transform(tf_num_teste)
tf_cat_teste = ohe.transform(X_teste[cat_col])
X_teste_transformado = np.concatenate((tf_num_teste, tf_cat_teste), axis=1)

# Avaliação no conjunto de dados de teste
y_pred_teste = model.predict(X_teste_transformado)

mse_teste = mean_squared_error(y_teste, y_pred_teste)
rmse_teste = np.sqrt(mse_teste)
r2_teste = r2_score(y_teste, y_pred_teste)

print("RMSE (teste):", rmse_teste)
print("R (teste):", r2_teste)

In [40]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#Criação de pipeline de pré-processamento para as colunas numéricas
num_preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy='mean')),
    ("scaler", StandardScaler())
])

#Criação do ColumnTransformer
full_preprocessing = ColumnTransformer([
    ("num", num_preprocessing, num_col),
    ("cat", OneHotEncoder(sparse_output=False, drop='first'), cat_col)
])

# Criação do pipeline final
final_pipeline = Pipeline([
    ("preprocessing", full_preprocessing),
    ("model", LinearRegression())
])



In [ ]:
#Treinar o modelo com pipeline
final_pipeline.fit(X_treinamento, y_treinamento)

In [ ]:
#Avaliação do pipeline

y_pred_treinamento = final_pipeline.predict(X_treinamento)

mse = mean_squared_error(y_treinamento, y_pred_treinamento)
rmse_treinamento = np.sqrt(mse)
r2_treinamento = r2_score(y_treinamento, y_pred_treinamento)

print("Avaliação do conjunto de treinamento:")
print("RMSE (treinamento):", rmse_treinamento)
print("R (Treinamento):", r2_treinamento)


# Avaliação no conjunto de dados de teste
y_pred_teste = final_pipeline.predict(X_teste)

mse_teste = mean_squared_error(y_teste, y_pred_teste)
rmse_teste = np.sqrt(mse_teste)
r2_teste = r2_score(y_teste, y_pred_teste)

print("\nAvaliação do conjunto de teste:")
print("RMSE (teste):", rmse_teste)
print("R (teste):", r2_teste)

In [48]:
from sklearn.model_selection import cross_val_score

#Validação cruzada com RMSE negativo com critério de pontuação
rmse_scores = cross_val_score(final_pipeline, X_treinamento, y_treinamento, scoring='neg_root_mean_squared_error', cv=4)
rmse_scores_abs = np.abs(rmse_scores)
print("Validação cruzada com RMSE:")
print("Scores:", rmse_scores_abs)
print("Média:", np.mean(rmse_scores_abs))

#Validação cruzada com R2 com critério de pontuação
r2_scores = cross_val_score(final_pipeline, X_treinamento, y_treinamento, cv=4, scoring='r2')
print("\nValidação cruzada com R2:")
print("Scores:", r2_scores)
print("Média:", np.mean(r2_scores))

[-0.2854804  -0.31458868 -0.30569181 -0.33011564]
Validação cruzada com RMSE:
Scores: [0.2854804  0.31458868 0.30569181 0.33011564]
Média: 0.30896913172043905

Validação cruzada com R2:
Scores: [0.8604119  0.8783167  0.86740351 0.82953801]
Média: 0.858917529995909
